# Pocket Duelist — fine-tune Qwen2.5-3B (QLoRA) → GGUF q4_k_m
Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.
The finished file is saved to Google Drive: `MyDrive/pocket_duelist/qwen2.5-3b-instruct.Q4_K_M.gguf`.
Copy it to `C:\pokemon\models\` on your PC.

In [ ]:
# 1) Mount Google Drive first (so the result survives a disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/pocket_duelist'
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 2) Install Unsloth
!pip install -q unsloth

In [ ]:
# 3) Load base model in 4-bit + add LoRA adapters
from unsloth import FastLanguageModel
MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)

In [ ]:
# 4) Dataset: 1,616 ChatML scenarios from the GitHub repo
from datasets import load_dataset
DATA_URL = 'https://raw.githubusercontent.com/chokunkie02/pokemon/main/backend/data/sft_data.jsonl'
ds = load_dataset('json', data_files=DATA_URL, split='train')
ds = ds.map(lambda b: {'text': [tokenizer.apply_chat_template(m, tokenize=False) for m in b['messages']]},
            batched=True, remove_columns=ds.column_names)
print(len(ds)); print(ds[0]['text'][:500])

In [ ]:
# 5) Train: 1 epoch, lr 5e-5 (loss only on the assistant JSON)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=MAX_SEQ,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=1, learning_rate=5e-5, warmup_steps=10,
        lr_scheduler_type='linear', optim='adamw_8bit', weight_decay=0.01,
        logging_steps=10, seed=3407, output_dir='outputs', report_to='none',
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
trainer.train()

In [ ]:
# 6) Quick sanity check on one training prompt
import json
FastLanguageModel.for_inference(model)
raw = load_dataset('json', data_files=DATA_URL, split='train')[0]['messages']
prompt = tokenizer.apply_chat_template(raw[:2], tokenize=False, add_generation_prompt=True)
ids = tokenizer(prompt, return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=128, temperature=0.1)
print('MODEL :', tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
print('TARGET:', raw[2]['content'])

In [ ]:
# 7) Merge + export GGUF q4_k_m, copy to Drive
import glob, shutil
model.save_pretrained_gguf('gguf_out', tokenizer, quantization_method='q4_k_m')
ggufs = [p for p in glob.glob('/content/**/*.gguf', recursive=True) if 'q4_k_m' in p.lower()]
print(ggufs)
dst = os.path.join(OUT_DIR, 'qwen2.5-3b-instruct.Q4_K_M.gguf')
shutil.copy(ggufs[0], dst)
print('DONE ->', dst, round(os.path.getsize(dst)/1e9, 2), 'GB')